# Análise de Volumetria e Estimativa de Custos — ClickHouse Cloud

## Arquitetura de Custos do ClickHouse Cloud

O ClickHouse Cloud cobra em **4 dimensões**:

| Dimensão | O que mede | Precificação |
|----------|-----------|-------------|
| **Storage** | Dados comprimidos em disco | ~$25.30/TiB/mês |
| **Compute** | CPU/RAM durante queries e ingest | $0.22 ~ $0.39/unidade/hora |
| **Data Transfer** | Egress (saída de dados) | Varia por região |
| **ClickPipes** | Ingestão via pipelines | Por volume ingerido |

### Fontes de Preço
- Storage: $25.30/TiB/mês (GCP)
- Compute: ~$0.2181/unidade/hora (Scale tier, GCP)
- Câmbio: ~R$ 5.22 / USD (Fev/2026)

### O que este notebook faz
1. Coleta volumetria real de TODAS as tabelas no ClickHouse
2. Coleta métricas de queries do `system.query_log`
3. Estima custo de **storage** em USD e BRL
4. Estima custo de **compute** baseado em queries executadas
5. Cria database `observability` com tabelas de custo (por hora, dia, mês)
6. Gera relatório consolidado

In [37]:
# ============================================================
# 1. IMPORTS E CONFIGURAÇÃO
# ============================================================
import warnings
warnings.filterwarnings('ignore')

import clickhouse_connect
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from typing import Dict, List, Any

# Configuração ClickHouse
CH_HOST = "e1a1lieug8.us-central1.gcp.clickhouse.cloud"
CH_PORT = 8443
CH_USER = "default"
CH_PASSWORD = "_uv765EvWphL_"

# Conectar
client = clickhouse_connect.get_client(
    host=CH_HOST,
    port=CH_PORT,
    username=CH_USER,
    password=CH_PASSWORD,
    secure=True,
    connect_timeout=60,
    send_receive_timeout=300
)

version = client.query("SELECT version()").result_rows[0][0]
print(f"ClickHouse Cloud conectado — versão {version}")
print(f"Data da análise: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

ClickHouse Cloud conectado — versão 25.12.1.1370
Data da análise: 2026-02-10 21:37:30


In [38]:
# ============================================================
# 2. CONSTANTES DE PRECIFICAÇÃO (ClickHouse Cloud GCP - Scale)
# ============================================================

# Storage: $25.30 por TiB/mês (dados comprimidos)
# Fonte: https://clickhouse.com/pricing e https://quesma.com/blog/clickhouse-pricing/
STORAGE_USD_PER_TIB_MONTH = 25.30
STORAGE_USD_PER_GB_MONTH = STORAGE_USD_PER_TIB_MONTH / 1024  # ~$0.02471/GB

# Compute: $0.2181/unidade/hora (Scale tier, GCP)
# 1 unidade ≈ 8 GiB RAM
# Fonte: https://www.glassflow.dev/blog/best-managed-clickhouse-service-2025
COMPUTE_USD_PER_UNIT_HOUR = 0.2181

# Câmbio USD → BRL (Fev/2026)
# Fonte: https://www.xe.com/en-us/currencyconverter/convert/?Amount=1&From=USD&To=BRL
USD_TO_BRL = 5.22

# Horas por mês (média)
HOURS_PER_MONTH = 730
HOURS_PER_DAY = 24

print("TABELA DE PREÇOS CLICKHOUSE CLOUD (GCP - Scale Tier)")
print("=" * 60)
print(f"  Storage:  ${STORAGE_USD_PER_TIB_MONTH:.2f}/TiB/mês = ${STORAGE_USD_PER_GB_MONTH:.5f}/GB/mês")
print(f"  Compute:  ${COMPUTE_USD_PER_UNIT_HOUR:.4f}/unidade/hora")
print(f"  Câmbio:   1 USD = R$ {USD_TO_BRL:.2f}")
print("=" * 60)

TABELA DE PREÇOS CLICKHOUSE CLOUD (GCP - Scale Tier)
  Storage:  $25.30/TiB/mês = $0.02471/GB/mês
  Compute:  $0.2181/unidade/hora
  Câmbio:   1 USD = R$ 5.22


In [39]:
# ============================================================
# 3. VOLUMETRIA DE STORAGE — Todas as tabelas
# ============================================================

print("ANÁLISE DE VOLUMETRIA — CLICKHOUSE CLOUD")
print("=" * 80)

# Query compatível com ClickHouse Cloud (sem data_compressed_bytes em system.tables)
volumetria_query = """
                   SELECT t.database,
                          t.name                                   AS table_name,
                          t.total_rows,
                          t.total_bytes,
                          formatReadableSize(t.total_bytes)        AS size_readable,
                          t.engine,
                          p.compressed_bytes,
                          formatReadableSize(p.compressed_bytes)   AS compressed_readable,
                          p.uncompressed_bytes,
                          formatReadableSize(p.uncompressed_bytes) AS uncompressed_readable,
                          IF(p.compressed_bytes > 0,
                             ROUND(p.uncompressed_bytes / p.compressed_bytes, 2),
                             0)                                    AS compression_ratio
                   FROM system.tables t
                            LEFT JOIN (SELECT database,
                                              `table`,
                                              SUM(data_compressed_bytes)   AS compressed_bytes,
                                              SUM(data_uncompressed_bytes) AS uncompressed_bytes
                                       FROM system.parts
                                       WHERE active = 1
                                       GROUP BY database, `table`) p ON t.database = p.database AND t.name = p.`table`
                   WHERE t.database IN ('raw', 'default', 'trusted', 'gold')
                     AND t.total_rows > 0
                   ORDER BY t.total_bytes DESC \
                   """

volumetria_df = client.query_df(volumetria_query)

print(f"\nTotal de tabelas com dados: {len(volumetria_df)}")
print(f"Total de linhas: {volumetria_df['total_rows'].sum():,.0f}")
print(f"Total em disco (total_bytes): {volumetria_df['total_bytes'].sum():,.0f} bytes")
print(f"Total comprimido: {volumetria_df['compressed_bytes'].sum():,.0f} bytes")
print(f"Total descomprimido: {volumetria_df['uncompressed_bytes'].sum():,.0f} bytes")
print()

# Tabela formatada
display_cols = ['database', 'table_name', 'total_rows', 'compressed_readable',
                'uncompressed_readable', 'compression_ratio', 'engine']
print(volumetria_df[display_cols].to_string(index=False))
print("=" * 80)

ANÁLISE DE VOLUMETRIA — CLICKHOUSE CLOUD

Total de tabelas com dados: 144
Total de linhas: 49,678,052
Total em disco (total_bytes): 1,551,286,950 bytes
Total comprimido: 1,549,972,755 bytes
Total descomprimido: 24,986,970,236 bytes

database                      table_name  total_rows compressed_readable uncompressed_readable  compression_ratio                   engine
     raw ginf_tst_historico_solicitacoes    28780000          873.55 MiB             10.24 GiB              12.01          SharedMergeTree
 trusted              ginf_tst_contratos     3051329          194.49 MiB              1.61 GiB               8.48          SharedMergeTree
     raw              ginf_tst_contratos     6107641           55.89 MiB              2.97 GiB              54.33          SharedMergeTree
 default                         bistage     6107641           53.46 MiB              2.97 GiB              56.80          SharedMergeTree
 trusted                     siga_sf2030      429512           34.75 MiB

In [40]:
# ============================================================
# 4. ESTIMATIVA DE CUSTO DE STORAGE
# ============================================================

print("ESTIMATIVA DE CUSTO DE STORAGE")
print("=" * 80)

total_compressed_bytes = volumetria_df['compressed_bytes'].sum()
total_compressed_gb = total_compressed_bytes / (1024 ** 3)
total_compressed_tib = total_compressed_bytes / (1024 ** 4)

# Custo mensal de storage
storage_usd_month = total_compressed_tib * STORAGE_USD_PER_TIB_MONTH
storage_brl_month = storage_usd_month * USD_TO_BRL

# Custo diário
storage_usd_day = storage_usd_month / 30
storage_brl_day = storage_brl_month / 30

# Custo por hora
storage_usd_hour = storage_usd_month / HOURS_PER_MONTH
storage_brl_hour = storage_brl_month / HOURS_PER_MONTH

print(f"\nDados comprimidos em disco: {total_compressed_gb:.4f} GB ({total_compressed_tib:.6f} TiB)")
print(f"\n{'Período':<15} {'USD':>15} {'BRL':>15}")
print("-" * 45)
print(f"{'Por hora':<15} {'$' + f'{storage_usd_hour:.6f}':>15} {'R$' + f'{storage_brl_hour:.4f}':>15}")
print(f"{'Por dia':<15} {'$' + f'{storage_usd_day:.4f}':>15} {'R$' + f'{storage_brl_day:.4f}':>15}")
print(f"{'Por mês':<15} {'$' + f'{storage_usd_month:.4f}':>15} {'R$' + f'{storage_brl_month:.4f}':>15}")
print(f"{'Por ano':<15} {'$' + f'{storage_usd_month * 12:.2f}':>15} {'R$' + f'{storage_brl_month * 12:.2f}':>15}")

# Custo por database
print(f"\nCUSTO POR DATABASE (mensal)")
print("-" * 60)
for db in volumetria_df['database'].unique():
    db_data = volumetria_df[volumetria_df['database'] == db]
    db_compressed = db_data['compressed_bytes'].sum()
    db_gb = db_compressed / (1024 ** 3)
    db_tib = db_compressed / (1024 ** 4)
    db_usd = db_tib * STORAGE_USD_PER_TIB_MONTH
    db_brl = db_usd * USD_TO_BRL
    db_rows = db_data['total_rows'].sum()
    print(f"  {db:<15} | {db_rows:>12,.0f} linhas | {db_gb:>8.4f} GB | ${db_usd:>8.4f} | R${db_brl:>8.2f}")

print("=" * 80)

ESTIMATIVA DE CUSTO DE STORAGE

Dados comprimidos em disco: 1.4435 GB (0.001410 TiB)

Período                     USD             BRL
---------------------------------------------
Por hora              $0.000049        R$0.0003
Por dia                 $0.0012        R$0.0062
Por mês                 $0.0357        R$0.1862
Por ano                   $0.43          R$2.23

CUSTO POR DATABASE (mensal)
------------------------------------------------------------
  raw             |   37,815,808 linhas |   1.0596 GB | $  0.0262 | R$    0.14
  trusted         |    5,335,748 linhas |   0.3234 GB | $  0.0080 | R$    0.04
  default         |    6,116,152 linhas |   0.0525 GB | $  0.0013 | R$    0.01
  gold            |      410,344 linhas |   0.0080 GB | $  0.0002 | R$    0.00


In [41]:
# ============================================================
# 5. ESTIMATIVA DE CUSTO DE COMPUTE (baseado em query_log)
# ============================================================

print("⚡ ESTIMATIVA DE CUSTO DE COMPUTE")
print("=" * 80)

# Tentar ler métricas do query_log (últimos 7 dias)
try:
    compute_query = """
                    SELECT toDate(event_time)                                       AS query_date,
                           toHour(event_time)                                       AS query_hour,
                           count()                                                  AS total_queries,
                           SUM(read_rows)                                           AS total_read_rows,
                           SUM(read_bytes)                                          AS total_read_bytes,
                           SUM(written_rows)                                        AS total_written_rows,
                           SUM(written_bytes)                                       AS total_written_bytes,
                           SUM(memory_usage)                                        AS total_memory_usage,
                           AVG(query_duration_ms) / 1000                            AS avg_duration_sec,
                           SUM(query_duration_ms) / 1000                            AS total_duration_sec,
                           SUM(ProfileEvents['OSCPUVirtualTimeMicroseconds']) / 1e6 AS total_cpu_seconds
                    FROM system.query_log
                    WHERE event_time >= now() - INTERVAL 7 DAY
                      AND type = 'QueryFinish'
                      AND query_kind = 'Select'
                    GROUP BY query_date, query_hour
                    ORDER BY query_date DESC, query_hour DESC \
                    """

    compute_df = client.query_df(compute_query)

    if len(compute_df) > 0:
        print(f"Dados de compute dos últimos 7 dias: {len(compute_df)} registros (hora a hora)")
        print(f"Total de queries: {compute_df['total_queries'].sum():,.0f}")
        print(f"Total de linhas lidas: {compute_df['total_read_rows'].sum():,.0f}")
        print(f"Total de bytes lidos: {compute_df['total_read_bytes'].sum() / (1024 ** 3):.2f} GB")
        print(f"Tempo total CPU: {compute_df['total_cpu_seconds'].sum():,.2f} segundos")

        # Estimativa: 1 compute unit ≈ 8 GiB RAM, ~2 vCPU
        # Tempo efetivo de compute = CPU seconds / 3600 = compute-hours
        total_cpu_hours = compute_df['total_cpu_seconds'].sum() / 3600
        compute_usd_7d = total_cpu_hours * COMPUTE_USD_PER_UNIT_HOUR
        compute_brl_7d = compute_usd_7d * USD_TO_BRL

        compute_usd_day = compute_usd_7d / 7
        compute_brl_day = compute_usd_day * USD_TO_BRL
        compute_usd_month = compute_usd_day * 30
        compute_brl_month = compute_usd_month * USD_TO_BRL

        print(f"\n{'Período':<20} {'USD':>15} {'BRL':>15}")
        print("-" * 50)
        print(f"{'Últimos 7 dias':<20} {'$' + f'{compute_usd_7d:.4f}':>15} {'R$' + f'{compute_brl_7d:.4f}':>15}")
        print(f"{'Média por dia':<20} {'$' + f'{compute_usd_day:.4f}':>15} {'R$' + f'{compute_brl_day:.4f}':>15}")
        print(
            f"{'Estimativa mensal':<20} {'$' + f'{compute_usd_month:.4f}':>15} {'R$' + f'{compute_brl_month:.2f}':>15}")
    else:
        print("⚠Sem dados de query_log nos últimos 7 dias")
        compute_usd_month = 0
        compute_brl_month = 0

except Exception as e:
    print(f"Não foi possível acessar system.query_log: {str(e)[:100]}")
    print("   (Normal em alguns planos do ClickHouse Cloud)")
    compute_usd_month = 0
    compute_brl_month = 0

print("=" * 80)

⚡ ESTIMATIVA DE CUSTO DE COMPUTE
Dados de compute dos últimos 7 dias: 6 registros (hora a hora)
Total de queries: 39,952
Total de linhas lidas: 7,174,205,239
Total de bytes lidos: 195.89 GB
Tempo total CPU: 3,011.49 segundos

Período                          USD             BRL
--------------------------------------------------
Últimos 7 dias               $0.1824        R$0.9524
Média por dia                $0.0261        R$0.1361
Estimativa mensal            $0.7819          R$4.08


In [42]:
# ============================================================
# 6. CUSTO POR TABELA (detalhado)
# ============================================================

print("CUSTO DETALHADO POR TABELA")
print("=" * 100)

# Calcular custo por tabela
volumetria_df['compressed_gb'] = volumetria_df['compressed_bytes'] / (1024 ** 3)
volumetria_df['compressed_tib'] = volumetria_df['compressed_bytes'] / (1024 ** 4)
volumetria_df['storage_usd_month'] = volumetria_df['compressed_tib'] * STORAGE_USD_PER_TIB_MONTH
volumetria_df['storage_brl_month'] = volumetria_df['storage_usd_month'] * USD_TO_BRL
volumetria_df['storage_usd_day'] = volumetria_df['storage_usd_month'] / 30
volumetria_df['storage_brl_day'] = volumetria_df['storage_usd_day'] * USD_TO_BRL

print(f"{'Database':<12} {'Tabela':<35} {'Linhas':>12} {'Comprimido':>12} {'$/mês':>10} {'R$/mês':>10}")
print("-" * 100)

for _, row in volumetria_df.sort_values('compressed_bytes', ascending=False).iterrows():
    print(f"{row['database']:<12} {row['table_name']:<35} {row['total_rows']:>12,.0f} "
          f"{row['compressed_readable']:>12} ${row['storage_usd_month']:>9.4f} R${row['storage_brl_month']:>8.4f}")

print("-" * 100)
print(f"{'TOTAL':<48} {volumetria_df['total_rows'].sum():>12,.0f} "
      f"{'':>12} ${volumetria_df['storage_usd_month'].sum():>9.4f} "
      f"R${volumetria_df['storage_brl_month'].sum():>8.4f}")
print("=" * 100)

CUSTO DETALHADO POR TABELA
Database     Tabela                                    Linhas   Comprimido      $/mês     R$/mês
----------------------------------------------------------------------------------------------------
raw          ginf_tst_historico_solicitacoes       28,780,000   873.55 MiB $   0.0211 R$  0.1100
trusted      ginf_tst_contratos                     3,051,329   194.49 MiB $   0.0047 R$  0.0245
raw          ginf_tst_contratos                     6,107,641    55.89 MiB $   0.0013 R$  0.0070
default      bistage                                6,107,641    53.46 MiB $   0.0013 R$  0.0067
trusted      siga_sf2030                              429,512    34.75 MiB $   0.0008 R$  0.0044
raw          siga_sf2030                              469,906    26.62 MiB $   0.0006 R$  0.0034
raw          ginf_tst_solicit_cadastradas             150,688    24.74 MiB $   0.0006 R$  0.0031
raw          sa1030                                   100,000    20.32 MiB $   0.0005 R$  0.0026

In [43]:
# ============================================================
# 7. CRIAR DATABASE E TABELAS DE OBSERVABILIDADE DE CUSTOS
# ============================================================

print("CRIANDO ESTRUTURA DE OBSERVABILIDADE DE CUSTOS")
print("=" * 80)

# Criar database
client.command("CREATE DATABASE IF NOT EXISTS observability")
print("Database 'observability' criada")

# Tabela: snapshot de custo por tabela (execução diária)
client.command("""
               CREATE TABLE IF NOT EXISTS observability.cost_snapshot_daily
               (
                   snapshot_date
                   Date,
                   database
                   String,
                   table_name
                   String,
                   total_rows
                   UInt64,
                   compressed_bytes
                   UInt64,
                   uncompressed_bytes
                   UInt64,
                   compression_ratio
                   Float32,
                   storage_usd_month
                   Float64,
                   storage_brl_month
                   Float64,
                   storage_usd_day
                   Float64,
                   storage_brl_day
                   Float64,
                   engine
                   String,
                   captured_at
                   DateTime
                   DEFAULT
                   now
               (
               )
                   ) ENGINE = ReplacingMergeTree
               (
                   captured_at
               )
                   PARTITION BY toYYYYMM
               (
                   snapshot_date
               )
                   ORDER BY
               (
                   snapshot_date,
                   database,
                   table_name
               )
               """)
print("Tabela 'observability.cost_snapshot_daily' criada")

# Tabela: métricas de compute por hora
client.command("""
               CREATE TABLE IF NOT EXISTS observability.compute_cost_hourly
               (
                   event_date
                   Date,
                   event_hour
                   UInt8,
                   total_queries
                   UInt64,
                   total_read_rows
                   UInt64,
                   total_read_bytes
                   UInt64,
                   total_written_rows
                   UInt64,
                   total_written_bytes
                   UInt64,
                   total_memory_bytes
                   UInt64,
                   avg_duration_sec
                   Float64,
                   total_cpu_seconds
                   Float64,
                   compute_usd_hour
                   Float64,
                   compute_brl_hour
                   Float64,
                   captured_at
                   DateTime
                   DEFAULT
                   now
               (
               )
                   ) ENGINE = ReplacingMergeTree
               (
                   captured_at
               )
                   PARTITION BY toYYYYMM
               (
                   event_date
               )
                   ORDER BY
               (
                   event_date,
                   event_hour
               )
               """)
print("Tabela 'observability.compute_cost_hourly' criada")

# Tabela: resumo de custo consolidado por dia
client.command("""
               CREATE TABLE IF NOT EXISTS observability.cost_summary_daily
               (
                   cost_date
                   Date,
                   total_tables
                   UInt32,
                   total_rows
                   UInt64,
                   total_compressed_gb
                   Float64,
                   storage_usd
                   Float64,
                   storage_brl
                   Float64,
                   compute_usd
                   Float64,
                   compute_brl
                   Float64,
                   total_usd
                   Float64,
                   total_brl
                   Float64,
                   total_queries
                   UInt64,
                   total_read_gb
                   Float64,
                   usd_to_brl_rate
                   Float64,
                   captured_at
                   DateTime
                   DEFAULT
                   now
               (
               )
                   ) ENGINE = ReplacingMergeTree
               (
                   captured_at
               )
                   PARTITION BY toYYYYMM
               (
                   cost_date
               )
                   ORDER BY cost_date
               """)
print("Tabela 'observability.cost_summary_daily' criada")

# View: custo mensal agregado
client.command("""
               CREATE
               OR REPLACE VIEW observability.v_cost_monthly AS
               SELECT toStartOfMonth(cost_date) AS month,
        COUNT(DISTINCT cost_date)            AS days_measured,
        MAX(total_tables)                    AS max_tables,
        MAX(total_rows)                      AS max_rows,
        MAX(total_compressed_gb)             AS max_compressed_gb,
        AVG(storage_usd)                     AS avg_storage_usd_day,
        SUM(storage_usd)                     AS total_storage_usd,
        SUM(storage_brl)                     AS total_storage_brl,
        SUM(compute_usd)                     AS total_compute_usd,
        SUM(compute_brl)                     AS total_compute_brl,
        SUM(total_usd)                       AS total_cost_usd,
        SUM(total_brl)                       AS total_cost_brl,
        SUM(total_queries)                   AS total_queries,
        SUM(total_read_gb)                   AS total_read_gb
               FROM observability.cost_summary_daily
               GROUP BY month
               ORDER BY month DESC
               """)
print("View 'observability.v_cost_monthly' criada")

client.command("""
CREATE TABLE IF NOT EXISTS observability.cost_forecast_daily
(forecast_date Date, point_estimate_usd Float64, point_estimate_brl Float64, lower_usd Float64, lower_brl Float64, upper_usd Float64, upper_brl Float64, captured_at DateTime DEFAULT now())
ENGINE = ReplacingMergeTree(captured_at) PARTITION BY toYYYYMM(forecast_date) ORDER BY forecast_date
""")
print("Tabela 'observability.cost_forecast_daily' criada")
for col in [('point_estimate_brl', 'point_estimate_usd'), ('lower_brl', 'lower_usd'), ('upper_brl', 'upper_usd')]:
    try:
        client.command(f"ALTER TABLE observability.cost_forecast_daily ADD COLUMN IF NOT EXISTS {col[0]} Float64 AFTER {col[1]}")
    except Exception:
        pass

client.command("""
CREATE OR REPLACE VIEW observability.v_cost_by_database_daily AS
SELECT snapshot_date AS cost_date,
       database,
       count() AS total_tables,
       sum(total_rows) AS total_rows,
       sum(compressed_bytes) / pow(1024, 3) AS compressed_gb,
       sum(storage_usd_day) AS storage_usd,
       sum(storage_brl_day) AS storage_brl
FROM observability.cost_snapshot_daily
GROUP BY snapshot_date, database
ORDER BY snapshot_date DESC, storage_usd DESC
""")
print("View 'observability.v_cost_by_database_daily' criada")

client.command("""
CREATE OR REPLACE VIEW observability.v_cost_month_comparison AS
WITH prev AS (
    SELECT toDayOfMonth(cost_date) AS dia, sum(total_usd) AS total_usd, sum(total_brl) AS total_brl
    FROM observability.cost_summary_daily
    WHERE cost_date >= toStartOfMonth(today() - 1) AND cost_date < toStartOfMonth(today())
    GROUP BY dia
),
curr AS (
    SELECT toDayOfMonth(cost_date) AS dia, sum(total_usd) AS total_usd, sum(total_brl) AS total_brl
    FROM observability.cost_summary_daily
    WHERE cost_date >= toStartOfMonth(today()) AND cost_date <= today()
    GROUP BY dia
),
all_dias AS (SELECT number AS dia FROM numbers(1, 31)),
joined AS (
    SELECT a.dia, coalesce(p.total_usd, 0) AS p_usd, coalesce(p.total_brl, 0) AS p_brl,
           coalesce(c.total_usd, 0) AS c_usd, coalesce(c.total_brl, 0) AS c_brl
    FROM all_dias a
    LEFT JOIN prev p ON a.dia = p.dia
    LEFT JOIN curr c ON a.dia = c.dia
)
SELECT dia,
       sum(p_usd) OVER (ORDER BY dia) AS mes_anterior_usd,
       sum(p_brl) OVER (ORDER BY dia) AS mes_anterior_brl,
       sum(c_usd) OVER (ORDER BY dia) AS mes_atual_usd,
       sum(c_brl) OVER (ORDER BY dia) AS mes_atual_brl
FROM joined
ORDER BY dia
""")
print("View 'observability.v_cost_month_comparison' criada")

client.command("""
CREATE TABLE IF NOT EXISTS observability.query_metric_hourly
(event_date Date, event_hour UInt8, total_queries UInt64, total_peak_memory_gb Float64,
 avg_memory_mb Float64, captured_at DateTime DEFAULT now())
ENGINE = ReplacingMergeTree(captured_at) PARTITION BY toYYYYMM(event_date) ORDER BY (event_date, event_hour)
""")
print("Tabela 'observability.query_metric_hourly' criada")

client.command("""
CREATE TABLE IF NOT EXISTS observability.part_log_daily
(event_date Date, database String, table_name String, event_type String, events UInt64,
 total_size_gb Float64, read_during_merge_gb Float64, captured_at DateTime DEFAULT now())
ENGINE = ReplacingMergeTree(captured_at) PARTITION BY toYYYYMM(event_date)
ORDER BY (event_date, database, table_name, event_type)
""")
print("Tabela 'observability.part_log_daily' criada")

client.command("""
CREATE TABLE IF NOT EXISTS observability.blob_storage_daily
(event_date Date, event_type String, operations UInt64, total_data_gb Float64,
 captured_at DateTime DEFAULT now())
ENGINE = ReplacingMergeTree(captured_at) PARTITION BY toYYYYMM(event_date) ORDER BY (event_date, event_type)
""")
print("Tabela 'observability.blob_storage_daily' criada")

print("\n" + "=" * 80)

CRIANDO ESTRUTURA DE OBSERVABILIDADE DE CUSTOS
Database 'observability' criada
Tabela 'observability.cost_snapshot_daily' criada
Tabela 'observability.compute_cost_hourly' criada
Tabela 'observability.cost_summary_daily' criada
View 'observability.v_cost_monthly' criada
Tabela 'observability.cost_forecast_daily' criada
View 'observability.v_cost_by_database_daily' criada


DatabaseError: :HTTPDriver for https://e1a1lieug8.us-central1.gcp.clickhouse.cloud:8443 returned response code 404)
 Code: 47. DB::Exception: Unknown expression identifier `dia` in scope WITH prev AS (SELECT toDayOfMonth(cost_date) AS dia, sum(total_usd) AS total_usd, sum(total_brl) AS total_brl FROM observability.cost_summary_daily WHERE (cost_date >= toStartOfMonth(today() - 1)) AND (cost_date < toStartOfMonth(today())) GROUP BY dia), curr AS (SELECT toDayOfMonth(cost_date) AS dia, sum(total_usd) AS total_usd, sum(total_brl) AS total_brl FROM observability.cost_summary_daily WHERE (cost_date >= toStartOfMonth(today())) AND (cost_date <= today()) GROUP BY dia), all_dias AS (SELECT number AS dia FROM numbers(1, 31)), joined AS (SELECT a.dia, coalesce(p.total_usd, 0) AS p_usd, coalesce(p.total_brl, 0) AS p_brl, coalesce(c.total_usd, 0) AS c_usd, coalesce(c.total_brl, 0) AS c_brl FROM all_dias AS a LEFT JOIN prev AS p ON a.dia = p.dia LEFT JOIN curr AS c ON a.dia = c.dia) SELECT dia, sum

In [ ]:
# ============================================================
# 8. INSERIR SNAPSHOT DE HOJE
# Todas as tabelas de custo (cost_snapshot_daily, cost_summary_daily,
# compute_cost_hourly, cost_forecast_daily) gravam sempre USD e BRL.
# ============================================================

print("INSERINDO SNAPSHOT DE CUSTOS DE HOJE")
print("=" * 80)

today = datetime.now().date()

# 8.1 — Inserir custo por tabela
cost_per_table = pd.DataFrame({
    'snapshot_date': today,
    'database': volumetria_df['database'],
    'table_name': volumetria_df['table_name'],
    'total_rows': volumetria_df['total_rows'].astype(np.uint64),
    'compressed_bytes': volumetria_df['compressed_bytes'].fillna(0).astype(np.uint64),
    'uncompressed_bytes': volumetria_df['uncompressed_bytes'].fillna(0).astype(np.uint64),
    'compression_ratio': volumetria_df['compression_ratio'].fillna(0).astype(np.float32),
    'storage_usd_month': volumetria_df['storage_usd_month'].astype(np.float64),
    'storage_brl_month': volumetria_df['storage_brl_month'].astype(np.float64),
    'storage_usd_day': volumetria_df['storage_usd_day'].astype(np.float64),
    'storage_brl_day': volumetria_df['storage_brl_day'].astype(np.float64),
    'engine': volumetria_df['engine']
})

client.insert_df('observability.cost_snapshot_daily', cost_per_table)
print(f"{len(cost_per_table)} registros inseridos em cost_snapshot_daily")

# 8.2 — Inserir resumo diário consolidado
total_queries_today = 0
total_read_gb_today = 0
compute_usd_today = 0
compute_brl_today = 0

try:
    today_compute = client.query(f"""
        SELECT
            count() as queries,
            SUM(read_bytes) / pow(1024, 3) as read_gb,
            SUM(ProfileEvents['OSCPUVirtualTimeMicroseconds']) / 1e6 / 3600 as cpu_hours
        FROM system.query_log
        WHERE event_date = today()
          AND type = 'QueryFinish'
    """).result_rows[0]

    total_queries_today = int(today_compute[0])
    total_read_gb_today = float(today_compute[1])
    cpu_hours_today = float(today_compute[2])
    compute_usd_today = cpu_hours_today * COMPUTE_USD_PER_UNIT_HOUR
    compute_brl_today = compute_usd_today * USD_TO_BRL
except Exception:
    pass

summary_today = pd.DataFrame([{
    'cost_date': today,
    'total_tables': np.uint32(len(volumetria_df)),
    'total_rows': np.uint64(volumetria_df['total_rows'].sum()),
    'total_compressed_gb': float(total_compressed_gb),
    'storage_usd': float(storage_usd_day),
    'storage_brl': float(storage_brl_day),
    'compute_usd': float(compute_usd_today),
    'compute_brl': float(compute_brl_today),
    'total_usd': float(storage_usd_day + compute_usd_today),
    'total_brl': float(storage_brl_day + compute_brl_today),
    'total_queries': np.uint64(total_queries_today),
    'total_read_gb': float(total_read_gb_today),
    'usd_to_brl_rate': float(USD_TO_BRL)
}])

client.insert_df('observability.cost_summary_daily', summary_today)
print(f"Resumo diário inserido em cost_summary_daily")

# 8.3 — Inserir compute por hora (hoje)
try:
    hourly_sql = """
    SELECT toDate(event_time) AS event_date,
           toHour(event_time) AS event_hour,
           count() AS total_queries,
           sum(read_rows) AS total_read_rows,
           sum(read_bytes) AS total_read_bytes,
           sum(written_rows) AS total_written_rows,
           sum(written_bytes) AS total_written_bytes,
           sum(memory_usage) AS total_memory_bytes,
           avg(query_duration_ms) / 1000 AS avg_duration_sec,
           sum(ProfileEvents['OSCPUVirtualTimeMicroseconds']) / 1e6 AS total_cpu_seconds
    FROM system.query_log
    WHERE event_date = today() AND type = 'QueryFinish'
    GROUP BY event_date, event_hour
    ORDER BY event_date, event_hour
    """
    hourly_rows = client.query(hourly_sql).result_rows
    if hourly_rows:
        compute_hourly = pd.DataFrame(hourly_rows, columns=[
            'event_date', 'event_hour', 'total_queries', 'total_read_rows', 'total_read_bytes',
            'total_written_rows', 'total_written_bytes', 'total_memory_bytes',
            'avg_duration_sec', 'total_cpu_seconds'
        ])
        cpu_hours = compute_hourly['total_cpu_seconds'].astype(float) / 3600
        compute_hourly['compute_usd_hour'] = (cpu_hours * COMPUTE_USD_PER_UNIT_HOUR).astype(np.float64)
        compute_hourly['compute_brl_hour'] = (compute_hourly['compute_usd_hour'] * USD_TO_BRL).astype(np.float64)
        compute_hourly['event_hour'] = compute_hourly['event_hour'].astype(np.uint8)
        compute_hourly['total_queries'] = compute_hourly['total_queries'].astype(np.uint64)
        compute_hourly['total_read_rows'] = compute_hourly['total_read_rows'].astype(np.uint64)
        compute_hourly['total_read_bytes'] = compute_hourly['total_read_bytes'].astype(np.uint64)
        compute_hourly['total_written_rows'] = compute_hourly['total_written_rows'].astype(np.uint64)
        compute_hourly['total_written_bytes'] = compute_hourly['total_written_bytes'].astype(np.uint64)
        compute_hourly['total_memory_bytes'] = compute_hourly['total_memory_bytes'].astype(np.uint64)
        compute_hourly['avg_duration_sec'] = compute_hourly['avg_duration_sec'].astype(np.float64)
        compute_hourly['total_cpu_seconds'] = compute_hourly['total_cpu_seconds'].astype(np.float64)
        client.insert_df('observability.compute_cost_hourly', compute_hourly)
        print(f"{len(compute_hourly)} registros inseridos em compute_cost_hourly")
    else:
        print("Nenhum registro de compute por hora hoje (query_log vazio)")
except Exception as e:
    print(f"Compute por hora não inserido: {e}")

try:
    qm_sql = """
    SELECT toDate(event_time) AS event_date, toHour(event_time) AS event_hour,
           count() AS total_queries,
           sum(peak_memory_usage) / 1024 / 1024 / 1024 AS total_peak_memory_gb,
           avg(memory_usage) / 1024 / 1024 AS avg_memory_mb
    FROM system.query_metric_log WHERE event_date = today()
    GROUP BY event_date, event_hour ORDER BY event_date, event_hour
    """
    qm_rows = client.query(qm_sql).result_rows
    if qm_rows:
        qm_df = pd.DataFrame(qm_rows, columns=['event_date', 'event_hour', 'total_queries', 'total_peak_memory_gb', 'avg_memory_mb'])
        qm_df['event_hour'] = qm_df['event_hour'].astype(np.uint8)
        qm_df['total_queries'] = qm_df['total_queries'].astype(np.uint64)
        client.insert_df('observability.query_metric_hourly', qm_df)
        print(f"{len(qm_df)} registros inseridos em query_metric_hourly")
except Exception as e:
    print(f"query_metric_hourly não inserido: {e}")

try:
    pl_sql = """
    SELECT toDate(event_time) AS event_date, database, table AS table_name, toString(event_type) AS event_type,
           count() AS events,
           sum(size_in_bytes) / 1024 / 1024 / 1024 AS total_size_gb,
           sum(read_bytes) / 1024 / 1024 / 1024 AS read_during_merge_gb
    FROM system.part_log WHERE event_date = today()
    GROUP BY event_date, database, table, event_type ORDER BY event_date, sum(read_bytes) DESC
    """
    pl_rows = client.query(pl_sql).result_rows
    if pl_rows:
        pl_df = pd.DataFrame(pl_rows, columns=['event_date', 'database', 'table_name', 'event_type', 'events', 'total_size_gb', 'read_during_merge_gb'])
        pl_df['events'] = pl_df['events'].astype(np.uint64)
        client.insert_df('observability.part_log_daily', pl_df)
        print(f"{len(pl_df)} registros inseridos em part_log_daily")
except Exception as e:
    print(f"part_log_daily não inserido: {e}")

try:
    bs_sql = """
    SELECT event_date, toString(event_type) AS event_type, count() AS operations,
           sum(data_size) / 1024 / 1024 / 1024 AS total_data_gb
    FROM system.blob_storage_log WHERE event_date = today()
    GROUP BY event_date, event_type ORDER BY event_date, sum(data_size) DESC
    """
    bs_rows = client.query(bs_sql).result_rows
    if bs_rows:
        bs_df = pd.DataFrame(bs_rows, columns=['event_date', 'event_type', 'operations', 'total_data_gb'])
        bs_df['operations'] = bs_df['operations'].astype(np.uint64)
        client.insert_df('observability.blob_storage_daily', bs_df)
        print(f"{len(bs_df)} registros inseridos em blob_storage_daily")
except Exception as e:
    print(f"blob_storage_daily não inserido: {e}")

print("=" * 80)

In [ ]:
# ============================================================
# 9. RELATÓRIO CONSOLIDADO FINAL
# ============================================================

print()
print("╔" + "═" * 78 + "╗")
print("║" + " RELATÓRIO DE CUSTOS — CLICKHOUSE CLOUD".center(78) + "║")
print("║" + f" {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}".center(78) + "║")
print("╠" + "═" * 78 + "╣")

print("║" + "".center(78) + "║")
print("║" + "  STORAGE".ljust(78) + "║")
print("║" + f"   Tabelas com dados:     {len(volumetria_df)}".ljust(78) + "║")
print("║" + f"   Total de linhas:       {volumetria_df['total_rows'].sum():,.0f}".ljust(78) + "║")
print("║" + f"   Dados comprimidos:     {total_compressed_gb:.4f} GB".ljust(78) + "║")
print("║" + f"   Taxa de compressão:    {volumetria_df['compression_ratio'].mean():.1f}x (média)".ljust(78) + "║")

print("║" + "".center(78) + "║")
print("╠" + "═" * 78 + "╣")
print("║" + "".center(78) + "║")
print("║" + "  CUSTOS ESTIMADOS".ljust(78) + "║")
print("║" + f"   Câmbio: 1 USD = R$ {USD_TO_BRL:.2f}".ljust(78) + "║")
print("║" + "".center(78) + "║")
print("║" + f"   {'':>5} {'Storage USD':>15} {'Storage BRL':>15} {'Total BRL':>15}".ljust(78) + "║")
print(
    "║" + f"   {'Hora':<5} {'$' + f'{storage_usd_hour:.6f}':>15} {'R$' + f'{storage_brl_hour:.4f}':>15} {'R$' + f'{storage_brl_hour:.4f}':>15}".ljust(
        78) + "║")
print(
    "║" + f"   {'Dia':<5} {'$' + f'{storage_usd_day:.4f}':>15} {'R$' + f'{storage_brl_day:.4f}':>15} {'R$' + f'{storage_brl_day:.4f}':>15}".ljust(
        78) + "║")
print(
    "║" + f"   {'Mês':<5} {'$' + f'{storage_usd_month:.4f}':>15} {'R$' + f'{storage_brl_month:.4f}':>15} {'R$' + f'{storage_brl_month:.4f}':>15}".ljust(
        78) + "║")
print(
    "║" + f"   {'Ano':<5} {'$' + f'{storage_usd_month * 12:.2f}':>15} {'R$' + f'{storage_brl_month * 12:.2f}':>15} {'R$' + f'{storage_brl_month * 12:.2f}':>15}".ljust(
        78) + "║")

print("║" + "".center(78) + "║")
print("╠" + "═" * 78 + "╣")
print("║" + "".center(78) + "║")
print("║" + " PROJEÇÃO (quando todas 54 tabelas estiverem migradas)".ljust(78) + "║")

# Estimar: hoje temos 4 tabelas com ~50k linhas ≈ X GB
# Se migrarmos todas 54 tabelas, estimar crescimento proporcional
current_tables = len(volumetria_df)
target_tables = 54
growth_factor = target_tables / max(current_tables, 1)

proj_gb = total_compressed_gb * growth_factor
proj_tib = proj_gb / 1024
proj_usd_month = proj_tib * STORAGE_USD_PER_TIB_MONTH
proj_brl_month = proj_usd_month * USD_TO_BRL

print("║" + f"   Tabelas atuais: {current_tables} → Alvo: {target_tables}".ljust(78) + "║")
print("║" + f"   Storage estimado: {proj_gb:.2f} GB ({growth_factor:.1f}x crescimento)".ljust(78) + "║")
print("║" + f"   Custo mensal estimado: ${proj_usd_month:.2f} / R${proj_brl_month:.2f}".ljust(78) + "║")
print("║" + f"   Custo anual estimado:  ${proj_usd_month * 12:.2f} / R${proj_brl_month * 12:.2f}".ljust(78) + "║")

print("║" + "".center(78) + "║")
print("╠" + "═" * 78 + "╣")
print("║" + "".center(78) + "║")
print("║" + "️  TABELAS DE CUSTOS CRIADAS".ljust(78) + "║")
print("║" + "   • observability.cost_snapshot_daily   (custo por tabela/dia)".ljust(78) + "║")
print("║" + "   • observability.compute_cost_hourly   (compute por hora)".ljust(78) + "║")
print("║" + "   • observability.cost_summary_daily    (resumo diário)".ljust(78) + "║")
print("║" + "   • observability.v_cost_monthly        (view mensal)".ljust(78) + "║")
print("║" + "   • observability.query_metric_hourly   (pico RAM por hora)".ljust(78) + "║")
print("║" + "   • observability.part_log_daily        (merges por tabela)".ljust(78) + "║")
print("║" + "   • observability.blob_storage_daily     (ops blob storage)".ljust(78) + "║")
print("║" + "".center(78) + "║")
print("╚" + "═" * 78 + "╝")

In [ ]:
# ============================================================
# QUANTO ESTOU PAGANDO HOJE? (visão completa)
# ============================================================
from datetime import datetime

print("╔" + "═" * 70 + "╗")
print("║" + " 💳 QUANTO PAGO HOJE — CLICKHOUSE CLOUD".center(70) + "║")
print("║" + f" {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}".center(70) + "║")
print("╠" + "═" * 70 + "╣")

# --- Custos calculados (variáveis já existentes) ---
storage_day_usd = storage_usd_day       # ~$0.0010
storage_day_brl = storage_brl_day       # ~R$0.0054
compute_day_usd = compute_usd_day       # ~$0.0247
compute_day_brl = compute_brl_day       # ~R$0.1288

# --- Custo mínimo do ClickHouse Cloud ---
# Scale tier GCP: serviço ativo = mínimo ~$0.34/hora (8GiB, 2 vCPU)
# Mas com auto-idle, o serviço pode pausar (custo $0 quando idle)
# O mínimo mensal para Scale tier é ~$196/mês se ligado 24/7
# Ref: https://clickhouse.com/pricing
MIN_ACTIVE_USD_HOUR = 0.34    # mínimo quando o serviço está "acordado"
IDLE_COST_USD_HOUR = 0.0      # serviço em idle = $0

# Vamos consultar quanto tempo o serviço ficou ativo hoje
try:
    uptime_query = """
        SELECT
            countDistinct(toStartOfMinute(event_time)) AS active_minutes
        FROM system.query_log
        WHERE event_date = today()
          AND type = 'QueryFinish'
    """
    active_minutes = client.query(uptime_query).result_rows[0][0]
    active_hours = active_minutes / 60
except Exception:
    active_hours = 1.0  # fallback: assumir 1 hora

# Custo de "wake-up" — quando o serviço sai do idle para atender queries
# ClickHouse Cloud cobra pelo tempo que o serviço fica ATIVO
wake_cost_usd = active_hours * MIN_ACTIVE_USD_HOUR
wake_cost_brl = wake_cost_usd * USD_TO_BRL

# Total real do dia
total_day_usd = storage_day_usd + wake_cost_usd
total_day_brl = total_day_usd * USD_TO_BRL

print("║" + "".center(70) + "║")
print("║" + " 📦 STORAGE (dados em disco)".ljust(70) + "║")
print("║" + f"   1.26 GB comprimidos → ${storage_day_usd:.4f} / R${storage_day_brl:.4f}".ljust(70) + "║")
print("║" + "".center(70) + "║")

print("║" + " ⚡ COMPUTE (serviço ativo)".ljust(70) + "║")
print("║" + f"   Serviço ativo hoje: ~{active_hours:.1f} hora(s)".ljust(70) + "║")
print("║" + f"   Custo mínimo ativo: ${MIN_ACTIVE_USD_HOUR:.2f}/hora".ljust(70) + "║")
print("║" + f"   Custo compute hoje: ${wake_cost_usd:.2f} / R${wake_cost_brl:.2f}".ljust(70) + "║")
print("║" + "".center(70) + "║")

print("╠" + "═" * 70 + "╣")
print("║" + "".center(70) + "║")
print("║" + " 💳 TOTAL DO DIA".ljust(70) + "║")
print("║" + f"   USD:  ${total_day_usd:.2f}".ljust(70) + "║")
print("║" + f"   BRL:  R${total_day_brl:.2f}".ljust(70) + "║")
print("║" + "".center(70) + "║")

print("╠" + "═" * 70 + "╣")
print("║" + "".center(70) + "║")
print("║" + " 📅 PROJEÇÃO MENSAL (se manter este padrão)".ljust(70) + "║")
total_month_usd = total_day_usd * 30
total_month_brl = total_month_usd * USD_TO_BRL
print("║" + f"   USD:  ${total_month_usd:.2f}/mês".ljust(70) + "║")
print("║" + f"   BRL:  R${total_month_brl:.2f}/mês".ljust(70) + "║")
print("║" + "".center(70) + "║")

print("╠" + "═" * 70 + "╣")
print("║" + "".center(70) + "║")
print("║" + " ⚠️  IMPORTANTE: O que REALMENTE aparece na fatura".ljust(70) + "║")
print("║" + "   O ClickHouse Cloud cobra COMPUTE pelo tempo que o".ljust(70) + "║")
print("║" + "   serviço fica ATIVO (não idle). Cada vez que você".ljust(70) + "║")
print("║" + "   roda uma query, o serviço 'acorda' e cobra ~$0.34/h".ljust(70) + "║")
print("║" + "   (mínimo). Storage é centavos. O grosso da conta".ljust(70) + "║")
print("║" + "   vem do COMPUTE.".ljust(70) + "║")
print("║" + "".center(70) + "║")
print("║" + " 💡 DICA: Configure auto-idle para 5 min no console".ljust(70) + "║")
print("║" + "   do ClickHouse Cloud para minimizar custos!".ljust(70) + "║")
print("║" + "".center(70) + "║")
print("╚" + "═" * 70 + "╝")

# --- Também ver direto na fatura ---
print("\n📌 Para ver a fatura REAL do ClickHouse Cloud:")
print("   1. Acesse: https://clickhouse.cloud/")
print("   2. Vá em: Organization → Billing → Usage")
print("   3. Lá mostra o consumo exato em USD por dia/hora")

In [ ]:
# ============================================================
# 💳 CUSTO REAL DE HOJE — Baseado nos SEUS dados reais
# ============================================================
from datetime import datetime

print("╔" + "═" * 72 + "╗")
print("║" + " 💳 CUSTO REAL HOJE — DADOS REAIS DO SEU CLUSTER".center(72) + "║")
print("║" + f" {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}".center(72) + "║")
print("╠" + "═" * 72 + "╣")

# ── 1. STORAGE REAL (dados comprimidos em disco) ──
storage_real = client.query("""
    SELECT
        SUM(data_compressed_bytes)   AS compressed,
        SUM(data_uncompressed_bytes) AS uncompressed
    FROM system.parts
    WHERE active = 1
      AND database IN ('raw', 'default', 'trusted', 'gold', 'observability')
""").result_rows[0]

compressed_bytes = storage_real[0]
uncompressed_bytes = storage_real[1]
compressed_gb = compressed_bytes / (1024 ** 3)
compressed_tib = compressed_bytes / (1024 ** 4)

# Preço real: $25.30/TiB/mês
storage_usd_mes = compressed_tib * 25.30
storage_usd_dia = storage_usd_mes / 30
storage_brl_dia = storage_usd_dia * 5.22
storage_brl_mes = storage_usd_mes * 5.22

print("║" + "".center(72) + "║")
print("║" + " 📦 STORAGE REAL".ljust(72) + "║")
print("║" + f"   Comprimido:  {compressed_gb:.4f} GB ({compressed_tib:.6f} TiB)".ljust(72) + "║")
print("║" + f"   Original:    {uncompressed_bytes / (1024**3):.2f} GB".ljust(72) + "║")
print("║" + f"   Compressão:  {uncompressed_bytes / max(compressed_bytes, 1):.1f}x".ljust(72) + "║")
print("║" + f"   Custo/dia:   ${storage_usd_dia:.4f} = R$ {storage_brl_dia:.4f}".ljust(72) + "║")
print("║" + f"   Custo/mês:   ${storage_usd_mes:.4f} = R$ {storage_brl_mes:.4f}".ljust(72) + "║")

# ── 2. COMPUTE REAL (tempo de serviço ativo HOJE) ──
try:
    compute_real = client.query("""
        SELECT
            countDistinct(toStartOfMinute(event_time)) AS minutos_ativos,
            count()                                    AS total_queries,
            SUM(read_bytes)                            AS bytes_lidos,
            SUM(query_duration_ms) / 1000              AS total_segundos,
            SUM(ProfileEvents['OSCPUVirtualTimeMicroseconds']) / 1e6 AS cpu_seg
        FROM system.query_log
        WHERE event_date = today()
          AND type = 'QueryFinish'
    """).result_rows[0]

    minutos_ativos = int(compute_real[0])
    total_queries = int(compute_real[1])
    bytes_lidos = float(compute_real[2])
    total_segundos = float(compute_real[3])
    cpu_seg = float(compute_real[4])

    # O ClickHouse Cloud cobra por tempo ATIVO (mínimo ~$0.34/h para 8GiB/2vCPU)
    horas_ativas = minutos_ativos / 60
    compute_usd_hoje = horas_ativas * 0.34
    compute_brl_hoje = compute_usd_hoje * 5.22

except Exception:
    minutos_ativos = 0
    total_queries = 0
    bytes_lidos = 0
    horas_ativas = 0
    compute_usd_hoje = 0
    compute_brl_hoje = 0

print("║" + "".center(72) + "║")
print("║" + " ⚡ COMPUTE REAL (hoje)".ljust(72) + "║")
print("║" + f"   Minutos ativos: {minutos_ativos} min ({horas_ativas:.2f}h)".ljust(72) + "║")
print("║" + f"   Queries hoje:   {total_queries:,}".ljust(72) + "║")
print("║" + f"   Dados lidos:    {bytes_lidos / (1024**3):.2f} GB".ljust(72) + "║")
print("║" + f"   Custo compute:  ${compute_usd_hoje:.2f} = R$ {compute_brl_hoje:.2f}".ljust(72) + "║")

# ── 3. TOTAL DO DIA ──
total_usd_hoje = storage_usd_dia + compute_usd_hoje
total_brl_hoje = total_usd_hoje * 5.22

print("║" + "".center(72) + "║")
print("╠" + "═" * 72 + "╣")
print("║" + "".center(72) + "║")
print("║" + " 💰 TOTAL REAL HOJE".ljust(72) + "║")
print("║" + f"   Storage:     ${storage_usd_dia:.4f}  +  Compute: ${compute_usd_hoje:.2f}".ljust(72) + "║")
print("║" + f"   ─────────────────────────────────────────".ljust(72) + "║")
print("║" + f"   HOJE:        ${total_usd_hoje:.2f} USD  =  R$ {total_brl_hoje:.2f}".ljust(72) + "║")
print("║" + "".center(72) + "║")

# ── 4. PROJEÇÃO MENSAL (baseada nos dados reais) ──
# Média de compute dos últimos 7 dias
try:
    media_7d = client.query("""
        SELECT
            countDistinct(toStartOfMinute(event_time)) / 7.0 AS avg_min_dia
        FROM system.query_log
        WHERE event_time >= now() - INTERVAL 7 DAY
          AND type = 'QueryFinish'
    """).result_rows[0]
    avg_min_dia = float(media_7d[0])
    avg_horas_dia = avg_min_dia / 60
except Exception:
    avg_horas_dia = horas_ativas

compute_usd_mes = avg_horas_dia * 0.34 * 30
compute_brl_mes = compute_usd_mes * 5.22
total_usd_mes = storage_usd_mes + compute_usd_mes
total_brl_mes = total_usd_mes * 5.22

print("╠" + "═" * 72 + "╣")
print("║" + "".center(72) + "║")
print("║" + " 📅 PROJEÇÃO MENSAL (média real dos últimos 7 dias)".ljust(72) + "║")
print("║" + f"   Média de uso diário: {avg_horas_dia:.1f}h/dia".ljust(72) + "║")
print("║" + f"   Storage/mês:  ${storage_usd_mes:.4f} = R$ {storage_brl_mes:.4f}".ljust(72) + "║")
print("║" + f"   Compute/mês:  ${compute_usd_mes:.2f} = R$ {compute_brl_mes:.2f}".ljust(72) + "║")
print("║" + f"   ─────────────────────────────────────────".ljust(72) + "║")
print("║" + f"   TOTAL/MÊS:   ${total_usd_mes:.2f} USD  =  R$ {total_brl_mes:.2f}".ljust(72) + "║")
print("║" + f"   TOTAL/ANO:   ${total_usd_mes * 12:.2f} USD  =  R$ {total_brl_mes * 12:.2f}".ljust(72) + "║")
print("║" + "".center(72) + "║")

# ── 5. BREAKDOWN POR DATABASE ──
print("╠" + "═" * 72 + "╣")
print("║" + "".center(72) + "║")
print("║" + " 🗄️  CUSTO POR DATABASE (storage mensal)".ljust(72) + "║")
print("║" + f"   {'Database':<15} {'GB':>10} {'$/mês':>10} {'R$/mês':>10}".ljust(72) + "║")
print("║" + f"   {'─' * 45}".ljust(72) + "║")

db_costs = client.query("""
    SELECT
        database,
        SUM(data_compressed_bytes) AS cb
    FROM system.parts
    WHERE active = 1
      AND database IN ('raw', 'default', 'trusted', 'gold', 'observability')
    GROUP BY database
    ORDER BY cb DESC
""").result_rows

for db_name, cb in db_costs:
    gb = cb / (1024 ** 3)
    tib = cb / (1024 ** 4)
    usd = tib * 25.30
    brl = usd * 5.22
    print("║" + f"   {db_name:<15} {gb:>10.4f} ${usd:>9.4f} R${brl:>9.4f}".ljust(72) + "║")

print("║" + "".center(72) + "║")
print("╠" + "═" * 72 + "╣")
print("║" + " ℹ️  Para ver a fatura oficial:".ljust(72) + "║")
print("║" + "    https://clickhouse.cloud → Billing → Usage".ljust(72) + "║")
print("╚" + "═" * 72 + "╝")

In [ ]:
# se precisar instalar no ambiente atual
%pip install -U dash plotly pandas statsmodels

In [ ]:
import numpy as np
import pandas as pd

def forecast_30d(
    daily_df: pd.DataFrame,
    value_col: str = "Total ($)",
    date_col: str = "Date",
    horizon: int = 30,
    alpha: float | None = None,
    beta: float | None = None,
    damped: bool = False,
    phi: float = 0.98,
) -> pd.DataFrame:
    """
    Forecast diário (horizon dias) sem statsmodels usando Holt (nível + tendência).

    - alpha (nível) e beta (tendência) podem ser fixos (0..1) ou None para auto-ajuste (grid simples).
    - damped=True aplica amortecimento na tendência com fator phi.
    Retorna um DataFrame com datas futuras e colunas: yhat, level, trend.
    """

    if horizon <= 0:
        raise ValueError("horizon deve ser > 0")

    s = daily_df[[date_col, value_col]].dropna().copy()
    s[date_col] = pd.to_datetime(s[date_col])
    s = s.sort_values(date_col).set_index(date_col)

    # Garantir frequência diária (preencher datas faltantes)
    s = s.asfreq("D")

    y = s[value_col].astype(float)

    # Preencher NaNs (se houver buracos)
    if y.isna().any():
        y = y.interpolate(method="time").ffill().bfill()

    y_values = y.to_numpy()
    n = len(y_values)
    if n < 3:
        # pouco histórico -> fallback: repetir último valor
        last = float(y_values[-1]) if n else 0.0
        future_idx = pd.date_range(y.index[-1] + pd.Timedelta(days=1), periods=horizon, freq="D")
        return pd.DataFrame(
            {"Date": future_idx, "yhat": np.full(horizon, last), "level": np.full(horizon, last), "trend": np.zeros(horizon)}
        )

    # Inicialização: nível = y0, tendência = y1 - y0 (simples e robusto)
    def holt_fit(yv: np.ndarray, a: float, b: float, damped_: bool, phi_: float):
        l = yv[0]
        t = yv[1] - yv[0]
        sse = 0.0
        for i in range(1, len(yv)):
            yhat = l + (phi_ * t if damped_ else t)
            err = yv[i] - yhat
            sse += err * err

            new_l = a * yv[i] + (1 - a) * (l + (phi_ * t if damped_ else t))
            new_t = b * (new_l - l) + (1 - b) * (phi_ * t if damped_ else t)

            l, t = new_l, new_t
        return l, t, sse

    def pick_params():
        # Grid pequeno para não ficar caro em notebook
        a_grid = np.linspace(0.05, 0.95, 19)
        b_grid = np.linspace(0.05, 0.95, 19)
        best = (None, None, np.inf, None, None)  # a, b, sse, level, trend
        for a in a_grid:
            for b in b_grid:
                l_, t_, sse_ = holt_fit(y_values, float(a), float(b), damped, float(phi))
                if sse_ < best[2]:
                    best = (float(a), float(b), float(sse_), float(l_), float(t_))
        return best

    if alpha is None or beta is None:
        a_opt, b_opt, _, l_last, t_last = pick_params()
    else:
        if not (0.0 < alpha < 1.0 and 0.0 < beta < 1.0):
            raise ValueError("alpha e beta devem estar entre 0 e 1 (exclusivo)")
        l_last, t_last, _ = holt_fit(y_values, float(alpha), float(beta), damped, float(phi))
        a_opt, b_opt = float(alpha), float(beta)

    # Previsão futura
    future_idx = pd.date_range(y.index[-1] + pd.Timedelta(days=1), periods=horizon, freq="D")

    yhat = np.empty(horizon, dtype=float)
    level = np.empty(horizon, dtype=float)
    trend = np.empty(horizon, dtype=float)

    l = float(l_last)
    t = float(t_last)

    for h in range(1, horizon + 1):
        # tendência amortecida soma uma progressão geométrica
        if damped:
            # l + t*(phi + phi^2 + ... + phi^h) = l + t*phi*(1-phi^h)/(1-phi)
            if abs(phi - 1.0) < 1e-12:
                yhat[h - 1] = l + t * h
            else:
                yhat[h - 1] = l + t * (phi * (1 - phi**h) / (1 - phi))
        else:
            yhat[h - 1] = l + t * h

        level[h - 1] = l
        trend[h - 1] = t

    out = pd.DataFrame(
        {
            "Date": future_idx,
            "yhat": yhat,
            "level": level,
            "trend": trend,
            "alpha": a_opt,
            "beta": b_opt,
        }
    )
    return out

In [ ]:
import numpy as np
import pandas as pd

def forecast_30d(
    daily_df: pd.DataFrame,
    value_col: str = "Total ($)",
    date_col: str = "Date",
    horizon: int = 30,
    alpha: float | None = None,
    beta: float | None = None,
    damped: bool = False,
    phi: float = 0.98,
) -> pd.DataFrame:
    """
    Forecast diário (horizon dias) sem statsmodels usando Holt (nível + tendência).

    - alpha (nível) e beta (tendência) podem ser fixos (0..1) ou None para auto-ajuste (grid simples).
    - damped=True aplica amortecimento na tendência com fator phi.
    Retorna um DataFrame com datas futuras e colunas: yhat, level, trend.
    """

    if horizon <= 0:
        raise ValueError("horizon deve ser > 0")

    s = daily_df[[date_col, value_col]].dropna().copy()
    s[date_col] = pd.to_datetime(s[date_col])
    s = s.sort_values(date_col).set_index(date_col)

    # Garantir frequência diária (preencher datas faltantes)
    s = s.asfreq("D")

    y = s[value_col].astype(float)

    # Preencher NaNs (se houver buracos)
    if y.isna().any():
        y = y.interpolate(method="time").ffill().bfill()

    y_values = y.to_numpy()
    n = len(y_values)
    if n < 3:
        # pouco histórico -> fallback: repetir último valor
        last = float(y_values[-1]) if n else 0.0
        future_idx = pd.date_range(y.index[-1] + pd.Timedelta(days=1), periods=horizon, freq="D")
        return pd.DataFrame(
            {"Date": future_idx, "yhat": np.full(horizon, last), "level": np.full(horizon, last), "trend": np.zeros(horizon)}
        )

    # Inicialização: nível = y0, tendência = y1 - y0 (simples e robusto)
    def holt_fit(yv: np.ndarray, a: float, b: float, damped_: bool, phi_: float):
        l = yv[0]
        t = yv[1] - yv[0]
        sse = 0.0
        for i in range(1, len(yv)):
            yhat = l + (phi_ * t if damped_ else t)
            err = yv[i] - yhat
            sse += err * err

            new_l = a * yv[i] + (1 - a) * (l + (phi_ * t if damped_ else t))
            new_t = b * (new_l - l) + (1 - b) * (phi_ * t if damped_ else t)

            l, t = new_l, new_t
        return l, t, sse

    def pick_params():
        # Grid pequeno para não ficar caro em notebook
        a_grid = np.linspace(0.05, 0.95, 19)
        b_grid = np.linspace(0.05, 0.95, 19)
        best = (None, None, np.inf, None, None)  # a, b, sse, level, trend
        for a in a_grid:
            for b in b_grid:
                l_, t_, sse_ = holt_fit(y_values, float(a), float(b), damped, float(phi))
                if sse_ < best[2]:
                    best = (float(a), float(b), float(sse_), float(l_), float(t_))
        return best

    if alpha is None or beta is None:
        a_opt, b_opt, _, l_last, t_last = pick_params()
    else:
        if not (0.0 < alpha < 1.0 and 0.0 < beta < 1.0):
            raise ValueError("alpha e beta devem estar entre 0 e 1 (exclusivo)")
        l_last, t_last, _ = holt_fit(y_values, float(alpha), float(beta), damped, float(phi))
        a_opt, b_opt = float(alpha), float(beta)

    # Previsão futura
    future_idx = pd.date_range(y.index[-1] + pd.Timedelta(days=1), periods=horizon, freq="D")

    yhat = np.empty(horizon, dtype=float)
    level = np.empty(horizon, dtype=float)
    trend = np.empty(horizon, dtype=float)

    l = float(l_last)
    t = float(t_last)

    for h in range(1, horizon + 1):
        # tendência amortecida soma uma progressão geométrica
        if damped:
            # l + t*(phi + phi^2 + ... + phi^h) = l + t*phi*(1-phi^h)/(1-phi)
            if abs(phi - 1.0) < 1e-12:
                yhat[h - 1] = l + t * h
            else:
                yhat[h - 1] = l + t * (phi * (1 - phi**h) / (1 - phi))
        else:
            yhat[h - 1] = l + t * h

        level[h - 1] = l
        trend[h - 1] = t

    out = pd.DataFrame(
        {
            "Date": future_idx,
            "yhat": yhat,
            "level": level,
            "trend": trend,
            "alpha": a_opt,
            "beta": b_opt,
        }
    )
    return out

In [ ]:
import numpy as np
import pandas as pd

def _holt_fit_forecast(y, alpha, beta, horizon):
    n = len(y)
    if n < 2:
        last = float(y[-1]) if n else 0.0
        return np.full(horizon, last), np.full(n, last), last, 0.0
    level, trend = float(y[0]), float(y[1] - y[0])
    fitted = np.empty(n)
    fitted[0] = level
    for i in range(1, n):
        fitted[i] = level + trend
        new_level = alpha * y[i] + (1 - alpha) * (level + trend)
        new_trend = beta * (new_level - level) + (1 - beta) * trend
        level, trend = new_level, new_trend
    yhat = np.array([level + trend * (h + 1) for h in range(horizon)])
    resid_std = np.nanstd(y - fitted) if n > 2 else (float(np.nanstd(y)) or 1e-6)
    return yhat, fitted, level, resid_std

def forecast_30d(daily_df: pd.DataFrame, value_col="Total ($)", date_col="Date", horizon=30):
    s = daily_df[[date_col, value_col]].dropna().copy()
    s[date_col] = pd.to_datetime(s[date_col])
    s = s.groupby(date_col, as_index=False)[value_col].sum()
    s = s.sort_values(date_col).set_index(date_col)[value_col].asfreq("D")

    # preenche dias faltantes (se houver) com interpolação leve
    s = s.interpolate(limit_direction="both")
    y = s.astype(float).to_numpy()
    if len(y) < 2:
        future_idx = pd.date_range(s.index[-1] + pd.Timedelta(days=1), periods=horizon, freq="D")
        last = float(y[-1]) if len(y) else 0.0
        return pd.DataFrame({"Date": future_idx, "yhat": last, "lower": last, "upper": last})
    best_sse, best_alpha, best_beta = np.inf, 0.3, 0.1
    for a in np.linspace(0.1, 0.95, 9):
        for b in np.linspace(0.05, 0.9, 9):
            _, fitted, _, _ = _holt_fit_forecast(y, a, b, 1)
            sse = np.nansum((y - fitted) ** 2)
            if sse < best_sse:
                best_sse, best_alpha, best_beta = sse, a, b
    yhat, _, _, resid_std = _holt_fit_forecast(y, best_alpha, best_beta, horizon)
    future_idx = pd.date_range(s.index[-1] + pd.Timedelta(days=1), periods=horizon, freq="D")
    r = resid_std if isinstance(resid_std, (int, float)) and resid_std > 0 else (float(np.nanstd(y)) or 1e-6)
    return pd.DataFrame({"Date": future_idx, "yhat": yhat, "lower": yhat - 1.96 * r, "upper": yhat + 1.96 * r})

try:
    _ = daily_df
except NameError:
    rows = client.query("SELECT cost_date, total_usd FROM observability.cost_summary_daily ORDER BY cost_date").result_rows
    daily_df = pd.DataFrame(rows, columns=["Date", "Total ($)"])
    daily_df["Date"] = pd.to_datetime(daily_df["Date"])
    daily_df = daily_df.groupby("Date", as_index=False)["Total ($)"].sum()

fc30 = forecast_30d(daily_df)
try:
    usd = fc30['yhat'].astype(float)
    low_usd = fc30['lower'].astype(float)
    up_usd = fc30['upper'].astype(float)
    forecast_df = pd.DataFrame({
        'forecast_date': pd.to_datetime(fc30['Date']).dt.date,
        'point_estimate_usd': usd,
        'point_estimate_brl': (usd * USD_TO_BRL).astype(float),
        'lower_usd': low_usd,
        'lower_brl': (low_usd * USD_TO_BRL).astype(float),
        'upper_usd': up_usd,
        'upper_brl': (up_usd * USD_TO_BRL).astype(float)
    })
    client.insert_df('observability.cost_forecast_daily', forecast_df)
    print(f"Previsão Holt: {len(forecast_df)} dias em observability.cost_forecast_daily (USD + BRL)")
except Exception as e:
    print(f"Forecast não gravado: {e}")

## Dashboard de custos

Duas formas de visualizar os dados:

1. **Aqui no notebook (Dash)** — Execute a célula abaixo para carregar os dados do ClickHouse e em seguida a célula do Dash. O navegador abrirá um dashboard interativo (porta 8050) com abas: custo por dia, por mês, por database, armazenamento por tabela e previsão 30 dias.

2. **No ClickHouse** — No console do ClickHouse Cloud (SQL) use as tabelas e views: `observability.cost_summary_daily`, `observability.v_cost_monthly`, `observability.v_cost_by_database_daily`, `observability.v_cost_month_comparison`, `observability.cost_snapshot_daily`, `observability.compute_cost_hourly`, `observability.query_metric_hourly`, `observability.part_log_daily`, `observability.blob_storage_daily`. Para gráficos avançados, conecte um Grafana ao ClickHouse usando o mesmo endpoint. Ver `platform/docs/clickhouse-system-tables-cost-visibility.md`.

In [ ]:
# ============================================================
# CARREGAR DADOS DO DASHBOARD A PARTIR DO CLICKHOUSE
# Execute esta célula antes de rodar o Dash (próxima célula)
# ============================================================
import pandas as pd

daily_df = pd.DataFrame()
component_cols = []
by_service = pd.DataFrame(columns=["Entity Name", "Total ($)"])
storage_tbl = pd.DataFrame(columns=["database", "table_name", "storage_usd_day", "compressed_gb"])
storage_metric = "storage_usd_day"

try:
    daily_rows = client.query("""
        SELECT cost_date, total_usd, storage_usd, compute_usd
        FROM observability.cost_summary_daily
        ORDER BY cost_date
    """).result_rows
    if daily_rows:
        daily_df = pd.DataFrame(daily_rows, columns=["Date", "Total ($)", "Storage ($)", "Compute ($)"])
        daily_df["Date"] = pd.to_datetime(daily_df["Date"])
        component_cols = ["Storage ($)", "Compute ($)"]

    by_svc_rows = client.query("""
        SELECT database, sum(storage_usd) AS total_usd
        FROM observability.v_cost_by_database_daily
        GROUP BY database
        ORDER BY total_usd DESC
    """).result_rows
    if by_svc_rows:
        by_service = pd.DataFrame(by_svc_rows, columns=["Entity Name", "Total ($)"])

    snap_rows = client.query("""
        SELECT database, table_name, storage_usd_day, compressed_bytes / pow(1024, 3) AS compressed_gb
        FROM observability.cost_snapshot_daily
        WHERE snapshot_date = (SELECT max(snapshot_date) FROM observability.cost_snapshot_daily)
        ORDER BY storage_usd_day DESC
    """).result_rows
    if snap_rows:
        storage_tbl = pd.DataFrame(snap_rows, columns=["database", "table_name", "storage_usd_day", "compressed_gb"])
except Exception as e:
    print(f"Erro ao carregar dados: {e}")

if daily_df.empty:
    daily_df = pd.DataFrame([{"Date": pd.Timestamp("today").normalize(), "Total ($)": 0, "Storage ($)": 0, "Compute ($)": 0}])
    component_cols = []

print(f"daily_df: {len(daily_df)} dias | by_service: {len(by_service)} databases | storage_tbl: {len(storage_tbl)} tabelas")

In [ ]:
from dash import Dash, dcc, html, Input, Output
import plotly.express as px
import plotly.graph_objects as go

app = Dash(__name__)

min_date = daily_df["Date"].min()
max_date = daily_df["Date"].max()

app.layout = html.Div(
    style={"fontFamily": "Arial", "maxWidth": "1200px", "margin": "0 auto", "padding": "16px"},
    children=[
        html.H2("Dashboard de Custos — ClickHouse Cloud"),
        html.Div(
            style={"display": "flex", "gap": "16px", "alignItems": "center", "flexWrap": "wrap"},
            children=[
                html.Div(
                    children=[
                        html.Label("Período"),
                        dcc.DatePickerRange(
                            id="date-range",
                            min_date_allowed=min_date,
                            max_date_allowed=max_date,
                            start_date=max_date - pd.Timedelta(days=30),
                            end_date=max_date,
                            display_format="YYYY-MM-DD",
                        ),
                    ]
                ),
                html.Div(
                    children=[
                        html.Label("Top N tabelas (storage)"),
                        dcc.Slider(
                            id="topn-storage",
                            min=5, max=50, step=5, value=15,
                            marks={i: str(i) for i in [5, 10, 15, 20, 30, 50]}
                        )
                    ],
                    style={"minWidth": "320px"}
                ),
            ],
        ),
        html.Hr(),
        dcc.Tabs(
            id="tabs",
            value="tab-daily",
            children=[
                dcc.Tab(label="Custo por dia", value="tab-daily"),
                dcc.Tab(label="Custo por mês", value="tab-monthly"),
                dcc.Tab(label="Por serviços", value="tab-services"),
                dcc.Tab(label="Armazenamento", value="tab-storage"),
                dcc.Tab(label="Previsão (30 dias)", value="tab-forecast"),
            ],
        ),
        html.Div(id="tab-content", style={"paddingTop": "12px"})
    ],
)

def _filter_daily(start_date, end_date):
    m = (daily_df["Date"] >= pd.to_datetime(start_date)) & (daily_df["Date"] <= pd.to_datetime(end_date))
    return daily_df.loc[m].copy()

def _monthly_from_daily(df):
    tmp = df.copy()
    tmp["Month"] = tmp["Date"].dt.to_period("M").dt.to_timestamp()
    out = tmp.groupby("Month", as_index=False)["Total ($)"].sum()
    return out

@app.callback(
    Output("tab-content", "children"),
    Input("tabs", "value"),
    Input("date-range", "start_date"),
    Input("date-range", "end_date"),
    Input("topn-storage", "value"),
)
def render_tab(tab, start_date, end_date, topn_storage):
    ddf = _filter_daily(start_date, end_date)

    if tab == "tab-daily":
        # 1) Linha do total
        fig_total = px.line(ddf, x="Date", y="Total ($)", title="Custo total por dia (USD)")
        fig_total.update_layout(height=360)

        # 2) Área empilhada (componentes)
        if component_cols:
            fig_stack = go.Figure()
            for c in component_cols:
                fig_stack.add_trace(go.Scatter(
                    x=ddf["Date"], y=ddf[c],
                    mode="lines",
                    stackgroup="one",
                    name=c.replace(" ($)", ""),
                ))
            fig_stack.update_layout(
                title="Composição do custo por dia (USD) — área empilhada",
                height=420,
                yaxis_title="USD",
                xaxis_title="Data",
            )
        else:
            fig_stack = go.Figure().update_layout(
                title="Sem colunas de componentes para empilhar",
                height=200
            )

        return html.Div([
            dcc.Graph(figure=fig_total),
            dcc.Graph(figure=fig_stack),
        ])

    if tab == "tab-monthly":
        mdf = _monthly_from_daily(ddf)
        fig = px.bar(mdf, x="Month", y="Total ($)", title="Custo total por mês (USD)")
        fig.update_layout(height=420)
        return html.Div([dcc.Graph(figure=fig)])

    if tab == "tab-services":
        bf = by_service.head(30).copy()
        if bf.empty:
            fig = go.Figure().add_annotation(text="Sem dados de custo por database.", x=0.5, y=0.5, showarrow=False)
            fig.update_layout(height=400, title="Custo por database (storage USD)")
        else:
            fig = px.bar(
                bf.sort_values("Total ($)", ascending=True),
                x="Total ($)", y="Entity Name",
                orientation="h",
                title="Top 30 databases por custo de storage (USD)",
            )
            fig.update_layout(height=650, yaxis_title="")
        return html.Div([dcc.Graph(figure=fig)])

    if tab == "tab-storage":
        st = storage_tbl.copy()
        if st.empty:
            fig = go.Figure().add_annotation(text="Sem dados de snapshot de storage.", x=0.5, y=0.5, showarrow=False)
            fig.update_layout(height=400)
            return html.Div([dcc.Graph(figure=fig)])
        topn = st.head(int(topn_storage)).copy()
        topn["table_full"] = topn["database"].astype(str) + "." + topn["table_name"].astype(str)
        fig_top = px.bar(
            topn.sort_values(storage_metric, ascending=True),
            x=storage_metric, y="table_full",
            orientation="h",
            title=f"Top {topn_storage} tabelas por storage ({storage_metric})",
        )
        fig_top.update_layout(height=650, yaxis_title="")
        tm = st.head(200).copy()
        fig_tm = px.treemap(
            tm,
            path=["database", "table_name"],
            values=storage_metric,
            title=f"Treemap de storage por database/tabela (top 200)",
        )
        fig_tm.update_layout(height=520)
        return html.Div([dcc.Graph(figure=fig_top), dcc.Graph(figure=fig_tm)])

    if tab == "tab-forecast":
        # Previsão baseada no período filtrado (melhor: use histórico maior possível)
        fc = forecast_30d(ddf)

        fig = go.Figure()
        fig.add_trace(go.Scatter(x=ddf["Date"], y=ddf["Total ($)"], mode="lines", name="Histórico"))
        fig.add_trace(go.Scatter(x=fc["Date"], y=fc["yhat"], mode="lines", name="Previsão (yhat)"))
        fig.add_trace(go.Scatter(
            x=pd.concat([fc["Date"], fc["Date"][::-1]]),
            y=pd.concat([fc["upper"], fc["lower"][::-1]]),
            fill="toself",
            line=dict(color="rgba(0,0,0,0)"),
            name="Intervalo (~95%)",
            opacity=0.2
        ))
        fig.update_layout(
            title="Previsão de custo total — próximos 30 dias (USD)",
            height=520,
            yaxis_title="USD",
            xaxis_title="Data"
        )
        return html.Div([dcc.Graph(figure=fig)])

    return html.Div("Selecione uma aba.")

# Rodar o servidor
app.run(debug=True, port=8050)